# 2023년 제7차 근로환경조사(KWCS) 분석

## 단계: 01. 데이터 전처리 (Data Preprocessing)
- 목표: 5만 명 규모의 설문 원시자료를 분석 가능한 형태로 정제
- 데이터: 산업안전보건연구원 제7차 근로환경조사(2023), 50,195명 × 439개 변수
- 이전 프로젝트와의 차이: 지역 단위 집계 데이터 → 개인 단위 표본조사 마이크로데이터

### 1.1 원시자료 로드
- 확장자는 .csv이나 실제 구분자는 탭(\t)이며, 각 행 전체가 큰따옴표로 감싸져 있고 줄 끝에 쉼표 3개가 붙어 있음
- `quoting=csv.QUOTE_NONE`으로 따옴표를 무시하고 읽은 뒤, 첫/끝 컬럼의 잔여 문자를 제거
- `dtype=str`로 읽는 이유: 결측코드가 섞인 상태에서 숫자로 읽으면 컬럼별 타입 추론이 어긋남
- 공백 문자(' ')는 설문 미해당(skip pattern)을 의미하므로 결측(NA)으로 변환
- 원본 파일은 엑셀로 열지 않음. 엑셀은 저장 시 구분자를 쉼표로 바꾸고 유효숫자를 표시값 기준으로 잘라냄

In [1]:
import pandas as pd
import csv

filepath = r'C:\data\2023년 제7차 근로환경조사 원시자료.csv'

# 구분자는 탭, 인코딩은 cp949, 행 전체를 감싼 따옴표는 무시(QUOTE_NONE)
df = pd.read_csv(filepath, sep='\t', encoding='cp949',
                 quoting=csv.QUOTE_NONE, dtype=str)

# 첫/끝 컬럼에 붙은 따옴표와 꼬리 쉼표 제거
df.columns = [c.strip().strip('"').rstrip(',').strip('"') for c in df.columns]
first, last = df.columns[0], df.columns[-1]
df[first] = df[first].str.lstrip('"')
df[last]  = df[last].str.replace(r'",{0,3}$', '', regex=True)
df = df.replace(' ', pd.NA)

print(f'행 : {df.shape[0]}, 열 : {df.shape[1]}')   # 50195, 439
print(df.columns[:8].tolist())

행 : 50195, 열 : 439
['id', 'wt1', 'wt2', 'wt3', 'area', 'gender', 'year', 'age']


In [2]:
# 눈으로 보기 (엑셀 대신)
pd.set_option('display.max_columns', 30)
print(df.iloc[:10, :15])

          id          wt1               wt2               wt3 area gender  \
0  1000019_1  897.1816327  675.720677270798  1.18184603629422    1      1   
1  1000019_2  897.1816327  501.450045059828  .877043974068019    1      2   
2  1000042_1  897.1816327   568.48965743281  .994297304952782    1      1   
3  1000042_2  897.1816327  555.526955700343  .971625336819356    1      2   
4  1000049_1  897.1816327  387.349267596592  .677479929161676    1      1   
5  1000051_1  897.1816327  569.548473872926  .996149191471881    1      1   
6  1000053_1  897.1816327  331.675803102638  .580106168742357    1      1   
7  1000056_1  897.1816327  692.288622169203  1.21082363112942    1      1   
8  1000057_1  897.1816327    414.1639293313  .724379192054935    1      1   
9  1000057_2  897.1816327  534.554037506525  .934943374774034    1      2   

   year age estat country country_etc emp_type selfemp_be selfemp_be_etc  \
0  1989  34     2       1        <NA>        3       <NA>           <NA>   


### 1.2 파일 무결성 검증
- 5만 행 규모에서는 데이터 오염을 눈으로 확인할 수 없으므로 수치로 대조
- 코드북에 변수별 빈도가 기재되어 있어 이를 정답지로 사용
- 검증 항목: emp_type, gender, satisfaction의 값별 빈도 및 wt3의 분포
- 하나라도 어긋나면 파일 손상으로 판단하고 재다운로드

In [3]:
# 코드북 기재값과 대조 — 하나라도 어긋나면 파일이 오염된 것
expected = {
    'emp_type'    : {'1': 12867, '2': 3138, '3': 30150, '4': 4040},
    'gender'      : {'1': 23678, '2': 26517},
    'satisfaction': {'1': 2379, '2': 37688, '3': 8458,
                     '4': 1372, '8': 275, '9': 23},
}

for var, exp in expected.items():
    got = df[var].value_counts().sort_index().to_dict()
    print(f'{var:13s} {"통과" if got == exp else "불일치"}')

w = pd.to_numeric(df['wt3'], errors='coerce')
print(f'wt3 평균 {w.mean():.4f} (기대 1.0000), 최대 {w.max():.4f} (기대 20.7537)')

emp_type      통과
gender        통과
satisfaction  통과
wt3 평균 1.0000 (기대 1.0000), 최대 20.7537 (기대 20.7537)


> **검증 통과** <br>
> 세 변수의 빈도와 가중치 분포가 코드북 기재값과 일치. 원본 무결성 확인됨.

### 1.3 분석 모집단 확정 및 변수 축소
- 이 조사는 종사상지위(emp_type)에 따라 문항이 분기됨
- 임금근로자(emp_type=3)로 한정하면 공통 문항에 더해 Q11~Q26(고용형태, 상사 자질, 사업장 평가, 노조 유무)을 사용할 수 있음
- 판단이 필요 없는 변수부터 기계적으로 제거: 주관식 기타(_etc), 동거 가구원 정보(hm_), 타 지위 전용 문항(selfemp_/semp_/unfw_), 결측률 90% 초과 문항
- 설계 정보(id, wt1~wt3, stratification, district, household)는 복합표본 분석에 필요하므로 보존

### 1.4 결측코드 처리
- 무응답 유형별 코드: 7/77/777 = 해당없음, 8/88/888 = 모름·무응답, 9/99/999 = 거절
- 주의: 코드의 자릿수가 변수마다 다름. 유효값 범위와 겹치지 않도록 설계되어 있음
- 일괄 치환 금지. age의 77·88은 실제 나이(383명), ctime의 7·8·9는 실제 통근시간(분, 559명)
- 검산 기준: 전체 표본에서 ctime의 777/888/999를 제거하면 코드북 기재값(유효 47,335 / 평균 38.17 / 표준편차 32.361)과 일치해야 함

In [4]:
import pandas as pd
import numpy as np

# --- 01-1. 모집단 한정 ---
emp = df[df['emp_type'] == '3'].copy()
print(f'임금근로자 : {len(emp):,}명')

# --- 01-2. 기계적 축소 ---
drop_cols = (
    [c for c in emp.columns if c.endswith('_etc')]
  + [c for c in emp.columns if c.startswith('hm_')]
  + [c for c in emp.columns if c.startswith(('selfemp_', 'semp_', 'unfw_'))]
)
emp = emp.drop(columns=drop_cols)

# 임금근로자에게 전원 결측이거나 결측률 90% 넘는 열 제거
na_ratio = emp.isna().mean()
emp = emp.drop(columns=na_ratio[na_ratio > 0.90].index)
print(f'변수 : 439 → {emp.shape[1]}개')

# --- 01-3. 결측코드 처리 시범 (3개만) ---
emp['satisfaction'] = pd.to_numeric(emp['satisfaction'], errors='coerce')
emp.loc[emp['satisfaction'].isin([8, 9]), 'satisfaction'] = np.nan

emp['ctime'] = pd.to_numeric(emp['ctime'], errors='coerce')
emp.loc[emp['ctime'].isin([777, 888, 999]), 'ctime'] = np.nan

emp['age'] = pd.to_numeric(emp['age'], errors='coerce')   # 결측코드 없음, 그대로

print(f"\nsatisfaction 유효 : {emp['satisfaction'].notna().sum():,}")
print(f"ctime  평균 {emp['ctime'].mean():.2f} / 표준편차 {emp['ctime'].std():.3f}")
print(f"age    평균 {emp['age'].mean():.2f} / 최대 {emp['age'].max():.0f}")

임금근로자 : 30,150명
변수 : 439 → 291개

satisfaction 유효 : 30,012
ctime  평균 43.33 / 표준편차 29.674
age    평균 48.23 / 최대 93


> **가중치 재표준화 필요** <br>
> wt3는 전체 50,195명 기준으로 평균 1.0이 되도록 표준화된 값. 임금근로자만 추출하면 평균이 1.2802로 틀어지므로 부분집합 기준으로 재표준화해야 함.

### 1.5 결측코드 처리 — 변수군별 분기

설문 원시자료의 무응답은 유형별 숫자 코드로 기록되어 있다.
그러나 같은 숫자라도 변수에 따라 유효값일 수도, 결측일 수도 있으므로
일괄 치환이 불가능하다. 코드북의 '유효한 값 / 결측값' 구분선이 유일한 판단 근거다.


In [5]:
import numpy as np

KEEP_STR = ['id']                                   # 숫자 변환 금지

# A. 노출시간 척도 — 7=전혀 없음(유효), 결측은 8·9뿐
SCALE7 = ([f'hazard_phy{i}' for i in range(1,10)]
        + [f'hazard_erg{i}' for i in range(1,7)]
        + [f'hazard_psy{i}' for i in range(1,4)]
        + [f'useequip{i}'   for i in range(1,4)]
        + ['winten2_1','winten2_2'])

# D. 연속형·코드 변수 — 변수별 개별 지정, 빈 리스트는 '결측코드 없음'
NUMERIC = {
    'age':[], 'year':[], 'area':[], 'ind':[], 'ind2':[99], 'occ':[], 'occ2':[999],
    'wday_week':[], 'hh_num':[], 'eli_num':[], 'target':[], 'mode':[],
    'wtime_con2_day':[], 'wduration_y':[], 'comp_emp':[],
    'wtime_week':[], 'wtime_r':[], 'wtime_con_r':[], 'ptime_week':[],
    'edu':[8,9],                                    # B. 7=대학원 이상(유효)
    'earning2':[77,88,99], 'earning2_r':[77,88,99],
    'ctime':[777,888,999], 'earning1':[7777,8888,9999], 'earning1_r':[7777,8888,9999],
    'comp_size2':[88,99], 'comp_sizea_r':[88,99], 'comp_sizeb_r':[88,99],
    'emp_con_period_y':[77,88,99], 'emp_con_period_m':[77,88,99],
    'emp_con_period_r':[777,888,999],
    'heal_abs1':[888,999], 'job_c1':[666,888,999], 'job_c1_r':[888,999],
    'ptime_r':[888,999], 'woutside3_1':[888,999], 'woutside3_2':[888,999],
    'wt1':[], 'wt2':[], 'wt3':[],
    'stratification':[], 'district':[], 'household':[],
}

for c in emp.columns:
    if c in KEEP_STR:
        continue                                     # 문자열 그대로 보존
    emp[c] = pd.to_numeric(emp[c], errors='coerce')   # 컬럼 단위로 변환
    if   c in NUMERIC: codes = NUMERIC[c]
    elif c in SCALE7:  codes = [8, 9]
    else:              codes = [7, 8, 9]              # C. 일반 범주형
    if codes:
        emp.loc[emp[c].isin(codes), c] = np.nan

# --- 검산 ---
for c in ['id','hazard_phy1','occ','ind','area','edu','wsituation1','satisfaction']:
    s = emp[c]
    tail = '' if c == 'id' else f' / 최대 {s.max():.0f}'
    print(f'{c:14s} 유효 {s.notna().sum():6,}{tail}')

id             유효 30,150
hazard_phy1    유효 30,075 / 최대 7
occ            유효 30,150 / 최대 10
ind            유효 30,150 / 최대 21
area           유효 30,150 / 최대 17
edu            유효 30,123 / 최대 7
wsituation1    유효 28,074 / 최대 5
satisfaction   유효 30,012 / 최대 4


- **A. 노출시간 척도** (hazard_phy·erg·psy, useequip, winten2)
  - 1=근무시간 내내 ~ 7=전혀 없음. **7이 척도의 끝점이므로 유효값**
  - 결측은 8(모름/무응답), 9(거절)뿐
  - 7을 결측 처리하면 위험에 노출되지 않은 응답자가 전부 소실됨

- **B. 7 이상이 유효 범주인 변수** (edu, occ, ind, wday_week 등)
  - edu: 7=대학원 재학 이상 / occ: 8·9·10=장치조작·단순노무·군인
  - 범주 자체가 7을 넘어가므로 자릿수 기준 치환이 위험

- **C. 일반 범주형** (wsituation, wstat, emp_comp_ass, disc, wwa 등)
  - 유효값이 1-5 또는 1-2 범위. 7=해당없음, 8=모름/무응답, 9=거절
  - 7·8·9 모두 결측 처리

- **D. 연속형·코드 변수** (age, ctime, earning1, 사업장규모 등)
  - 결측코드의 자릿수가 유효값 범위와 겹치지 않게 설계되어 변수마다 다름
  - 예: age는 결측코드 없음(77·88은 실제 나이) / ctime은 777·888·999 / earning1은 7777·8888·9999
  - 변수별 개별 지정이 필요하며, 미확인 변수는 **손대지 않는 것이 기본값**

> **기본값 설계 원칙** <br>
> 초안에서는 미분류 변수에 [7,8,9]를 자동 적용했으나, 이로 인해 산업·직업 분류코드와
> 근속연수 등 연속형 변수 약 5만 셀이 조용히 소실되었다. 오류 메시지는 발생하지 않았다.
> 안전한 기본값은 '일괄 치환'이 아니라 '아무것도 하지 않음'이며,
> 처리 대상을 명시적으로 나열하는 화이트리스트 방식으로 전환했다.

> **검산 해석** <br>
> hazard_phy1 최대 7 — A그룹 분기가 작동, 노출 없음 응답이 보존됨 <br>
> occ 최대 10, ind 최대 21 — 분류코드가 잘리지 않음 <br>
> edu 최대 7 — 대학원 이상 학력 보존 <br>
> wsituation1 최대 5 — C그룹 분기가 작동, 해당없음(7)이 결측 처리됨 <br>
> id 유효 30,150 — 문자열 식별자가 숫자 변환에서 보호됨

### 1.6 구조적 결측 정리

결측률이 80%를 넘는 변수가 다수 존재하나, 이는 데이터 품질 문제가 아니라
설문의 분기(skip pattern) 구조가 반영된 결과다. 기계적으로 삭제하면
정보가 아니라 응답자를 잃는다.

In [6]:
import numpy as np

emp = emp.copy()                       # 조각난 블록 정리 (경고 해소)

# ① 누락됐던 결측코드 8개
EXTRA = {'wduration':[88,99], 'wday':[88,99], 'wtime':[888,999], 'ptime':[888,999],
         'wtime_night_a':[88,99], 'wtime_sun_a':[88,99],
         'wtime_sat_a':[88,99],   'wtime_long_a':[88,99]}
for c, codes in EXTRA.items():
    emp.loc[emp[c].isin(codes), c] = np.nan

# ② '안 함' 응답자의 일수는 결측이 아니라 0
for a, b in [('wtime_sun_a','wtime_sun'), ('wtime_sat_a','wtime_sat')]:
    emp.loc[(emp[a] == 0) & emp[b].isna(), b] = 0

# ③ 통합·계산 변수(_r)가 있으면 원본은 제거
REDUNDANT = ['earning1','earning2','earning2_r','wtime','wtime_week','wtime_month',
             'ptime','ptime_week','ptime_month','job_c1','wduration','wduration_y',
             'emp_con_period_y','emp_con_period_m','wtime_con1','wtime_con2',
             'wtime_con2_day','wtime_con2_week','wtime_con2_month','wday',
             'comp_size2','comp_size3','comp_size4']

# ④ 본문항이 있는 후속문항 제거
FOLLOWUP = ['ch_ic_a','ch_me_a','ch_ps_a','emp_suggest1_1','emp_suggest2_1','heal_lim2',
            'emp_con_renew','emp_tra1_1','emp_tra2_1','emp_tra_ass1','emp_tra_ask',
            'wteam2','wteam3_1','wteam3_2','wteam3_3','job_c2','job_c3','wtime_con3',
            'woutside3_1','woutside3_2','winterrupt2','heal_wsick2','emp_expect',
            'emp_expect_mon','emp_keep2','emp_expect2','wplace_sl',
            'wtime_night2_hours','wtime_night3_times']

# ⑤ 다문화가구·외국인 한정 문항 제거
FOREIGN = ['disc2','disc3','disc4']

drop = [c for c in REDUNDANT + FOLLOWUP + FOREIGN if c in emp.columns]
emp = emp.drop(columns=drop)
print(f'변수 : 291 → {emp.shape[1]}개')

# ⑥ 종속변수 결측 제거 → 그 다음에 가중치 재표준화 (순서 중요)
emp = emp[emp['satisfaction'].notna()].copy()
emp['wt_std'] = emp['wt3'] / emp['wt3'].mean()

print(f"최종 : {len(emp):,}명 × {emp.shape[1]}개 / wt_std 평균 {emp['wt_std'].mean():.4f}")
na = emp.isna().mean()
print(f'결측률 50% 초과 : {(na>0.5).sum()}개')

emp.to_csv(r'C:\data\kwcs2023_clean.csv', index=False, encoding='utf-8')
print(f'저장 완료 : {len(emp):,}행 × {emp.shape[1]}열')

변수 : 291 → 249개
최종 : 30,012명 × 250개 / wt_std 평균 1.0000
결측률 50% 초과 : 7개
저장 완료 : 30,012행 × 250열


- **본문항 / 후속문항 구조**
  - "변화가 있었는가"(본문항, 결측 2.6%) → "그 영향은"(후속문항, 결측 89.7%)
  - 후속문항은 본문항에서 특정 응답을 한 사람에게만 제시됨
  - 본문항을 유지하고 후속문항을 제거하면 정보 손실 없이 결측률이 정리됨

- **결측이 실제로는 0인 경우**
  - wtime_sun_a(일요일 근무 여부)=0인 26,656명은 wtime_sun(근무 일수)이 결측
  - 근무하지 않았으므로 일수를 묻지 않은 것. 결측이 아니라 0일에 해당
  - 0으로 복원하면 결측률 88.9% → 0.5%

- **통합·계산 변수 우선 사용**
  - 응답 단위(주/월)를 응답자가 선택하는 문항은 원본이 여러 컬럼으로 분리됨
  - 조사기관이 단위를 통일한 _r 변수(wtime_r, ptime_r, earning1_r 등)가 별도 제공됨
  - 예: earning1 결측 22.1% / earning2 81.2% vs earning1_r 3.4%
  - 원본 컬럼은 제거하고 _r 변수만 사용

- **누락 결측코드 보완**
  - wduration, wday, wtime, ptime 및 근무여부(_a) 변수 8개에서 88/99, 888/999 미처리 확인
  - ptime의 777은 '현재와 동일'을 뜻하는 유효 응답이므로 결측 처리 대상이 아님

> **가중치 재표준화 순서** <br>
> wt3는 전체 50,195명 기준 평균 1.0으로 표준화된 값이므로,
> 임금근로자 부분집합에서는 평균이 1.2802로 틀어진다.
> 재표준화는 분석 대상 행이 최종 확정된 뒤에 수행해야 평균이 정확히 1.0이 된다.

> **PerformanceWarning 대응** <br>
> 반복문으로 컬럼을 개별 대입하면 내부 메모리 블록이 조각나 경고가 발생한다.
> 계산 결과에는 영향이 없으며, 루프 종료 후 `emp = emp.copy()`로 해소한다.

In [9]:
# --- 2.1 종속변수 분포: 가중 vs 비가중 ---
s, w = emp['satisfaction'], emp['wt_std']
unw = s.value_counts(normalize=True).sort_index() * 100
wtd = s.groupby(s).apply(lambda g: w[g.index].sum()) / w.sum() * 100

lab = {1:'매우 만족', 2:'만족', 3:'별로 만족않음', 4:'전혀 만족않음'}
for k in [1, 2, 3, 4]:
    print(f'{k} {lab[k]:12s} 비가중 {unw[k]:6.2f}% 가중 {wtd[k]:6.2f}% ({wtd[k]-unw[k]:+.2f})')
print(f'불만족(3+4) : {unw[3]+unw[4]:.2f}% -> {wtd[3]+wtd[4]:.2f}%')

# --- 2.1 변수 유형 분류 ---
DESIGN = ['id','wt1','wt2','wt3','wt_std','stratification','district','household',
          'target','mode','emp_type','estat','country']
analysis = [c for c in emp.columns if c not in DESIGN]

rows = []
for c in analysis:
    nu = emp[c].dropna().nunique()
    t = ('이분형' if nu <= 2 else '순서형/소범주' if nu <= 7
    else '명목형(다범주)' if nu <= 25 else '연속형')
    rows.append((c, t, nu, round(emp[c].isna().mean(), 3)))

vtype = pd.DataFrame(rows, columns=['변수', '유형', '고유값', '결측률'])
print(f'\n분석 대상 {len(analysis)}개')
print(vtype['유형'].value_counts().to_string())

# --- 저분산 변수 탐지 (버리지 말고 목록만) ---
lowvar = [(c, round(emp[c].dropna().value_counts(normalize=True).iloc[0], 3))
          for c in analysis if len(emp[c].dropna()) > 0
          and emp[c].dropna().value_counts(normalize=True).iloc[0] > 0.95]
print(f'\n한 값에 95% 이상 몰린 변수 {len(lowvar)}개')

1 매우 만족        비가중   5.35% 가중   5.44% (+0.09)
2 만족           비가중  76.84% 가중  75.54% (-1.30)
3 별로 만족않음      비가중  15.14% 가중  16.18% (+1.03)
4 전혀 만족않음      비가중   2.66% 가중   2.84% (+0.18)
불만족(3+4) : 17.81% -> 19.02%

분석 대상 237개
유형
순서형/소범주     118
이분형         100
연속형          13
명목형(다범주)      6

한 값에 95% 이상 몰린 변수 25개
